# Reconstruction Impact: Spatial Diversity

This notebook computes non-overlapping Spatial Windows and compares Raw versus reconstructed effective diversity (`Neff`) on the full matched cohort. It also writes a separate Level1-derived anatomy assignment on the same grid.

**Evidence boundary.** Window diversity is a descriptive summary of inferred cluster composition. It does not prove that a reconstructed state is biologically new or spatially causal.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the REVISE repository root.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from revise.analysis.basic.spatial_region import (
    assign_anatomy_regions,
    assign_square_windows,
    compute_window_diversity,
)
from revise.analysis.reconstruction_impact import load_reconstruction_impact_config, run_partition_analysis

CONFIG_PATH = Path(os.environ.get(
    'REVISE_RECONSTRUCTION_IMPACT_CONFIG',
    REPO_ROOT / 'configs/analysis/reconstruction_impact_visiumhd_p1crc.yaml',
))
config = load_reconstruction_impact_config(CONFIG_PATH)
output_root = Path(os.environ.get('REVISE_ANALYSIS_OUTPUT_ROOT', REPO_ROOT / config['output']['dir']))
comparison_config = config['partition_change']['comparisons'][0]

In [ ]:
raw_full = ad.read_h5ad(REPO_ROOT / comparison_config['raw_h5ad'])
recon = ad.read_h5ad(REPO_ROOT / comparison_config['reconstructed_spatial_h5ad'])
if not recon.obs_names.isin(raw_full.obs_names).all():
    raise ValueError('The reconstructed spatial IDs are not all present in Raw.')
raw = raw_full[recon.obs_names].copy()

analysis = run_partition_analysis(
    raw, recon,
    level1_col=comparison_config['level1_column'],
    final_cluster_key=comparison_config.get('reconstructed_cluster_key'),
    resolution_mode=config['partition_change']['mode'],
    resolution_candidates=config['partition_change']['level1_resolution_candidates'],
    within_level1_resolution=config['partition_change']['within_level1_resolution'],
    random_state=config['partition_change']['random_state'],
    n_top_genes=config['partition_change']['n_top_genes'],
)
edge = 'raw_to_final_svc' if 'raw_to_final_svc' in analysis.comparisons else 'raw_to_recon_expression'
assignments = analysis.comparisons[edge].assignments.set_index('unit_id')
scale = config['spatial_region'].get('microns_per_coordinate', 1.0)
anatomy_coords = pd.DataFrame(
    raw_full.obsm[comparison_config['spatial_key']], index=raw_full.obs_names, columns=['x', 'y']
) * scale
grid_origin = (float(anatomy_coords['x'].min()), float(anatomy_coords['y'].min()))
window_side = config['spatial_region']['window_side_length_um']
anatomy_windows = assign_square_windows(
    anatomy_coords, window_side_length=window_side, origin=grid_origin
)
coords = pd.DataFrame(raw.obsm[comparison_config['spatial_key']], index=raw.obs_names, columns=['x', 'y']) * scale
windows = assign_square_windows(coords, window_side_length=window_side, origin=grid_origin)
metrics = compute_window_diversity(
    windows, assignments['raw_cluster'], assignments['recon_cluster'],
    min_units_per_window=config['spatial_region']['min_units_per_window'],
)
anatomy = assign_anatomy_regions(
    anatomy_windows, raw_full.obs[comparison_config['level1_column']],
    tumor_label=config['spatial_region']['anatomy_region']['tumor_label'],
    normal_source_label=config['spatial_region']['anatomy_region']['normal_source_label'],
)
metrics

In [ ]:
spatial_dir = output_root / 'spatial'
figure_dir = spatial_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
unit_window_map = windows.join(assignments[['raw_cluster', 'recon_cluster', 'unit_changed']])
unit_window_map['grid_origin_x'] = grid_origin[0]
unit_window_map['grid_origin_y'] = grid_origin[1]
unit_window_map.index.name = 'unit_id'
unit_window_map.to_csv(spatial_dir / 'unit_window_map.csv.gz', compression='gzip')
anatomy_windows.to_csv(
    spatial_dir / 'anatomy_window_map.csv.gz', compression='gzip', index_label='unit_id'
)
metrics.to_csv(spatial_dir / 'spatial_window_metrics.csv.gz', index=False, compression='gzip')
anatomy.to_csv(spatial_dir / 'anatomy_region_map.csv.gz', index=False, compression='gzip')

centers = windows.groupby('window_id')[['x', 'y']].mean().reset_index()
plot_data = centers.merge(metrics, on='window_id')
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for axis, column, title, cmap in zip(
    axes, ['neff_raw', 'neff_recon', 'delta_neff'],
    ['Raw Neff', 'Reconstructed Neff', 'Delta Neff'], ['viridis', 'viridis', 'coolwarm'],
):
    values = plot_data[column]
    kwargs = {'cmap': cmap}
    if column == 'delta_neff':
        limit = np.nanmax(np.abs(values))
        kwargs.update(vmin=-limit, vmax=limit)
    image = axis.scatter(plot_data['x'], plot_data['y'], c=values, s=12, **kwargs)
    axis.set_title(title)
    axis.set_aspect('equal')
    fig.colorbar(image, ax=axis)
fig.savefig(figure_dir / 'spatial_diversity_panels.png', dpi=180)
plt.show()

anatomy_centers = anatomy_windows.groupby('window_id')[['x', 'y']].mean().reset_index()
anatomy_plot = anatomy_centers.merge(anatomy[['window_id', 'region_label']], on='window_id')
colors = {'Tumor': '#bcbd22', 'Normal': '#4c78a8', 'Interface': '#e45756', 'Other': '#d9d9d9'}
fig, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
axis.scatter(anatomy_plot['x'], anatomy_plot['y'], c=anatomy_plot['region_label'].map(colors), s=12)
axis.set_title('Level1 anatomy regions')
axis.set_aspect('equal')
fig.savefig(figure_dir / 'anatomy_region_map.png', dpi=180)
plt.show()